# DrugMechDB association types: bootstrap evaluation against MechRepoNet

For every DrugMechDB (DMDB) *concept-type association* (e.g. `Protein_BiologicalProcess`),
this notebook measures what % of those DMDB edges are also present in MechRepoNet (MRN), then
compares that observed overlap against a **bootstrap null** built by randomly re-pairing the
heads and tails within each association type.



The bootstrap is stochastic, so values vary slightly between runs.

## 1. Setup

In [ ]:
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import yaml
from tqdm.auto import tqdm

# --- Inputs ---
DMDB_YAML  = "/home/agonzalez/DMDB_Analysis/0_data/external/indication_paths.yaml"
MRN_NODES  = "/home/agonzalez/data_KG/MIND_nodes/nodes.csv"
MRN_EDGES  = "/home/agonzalez/data_KG/MIND_edges/edges.csv"

# --- Output ---
OUT_CSV    = "dmdb_mrn_association_bootstrap.csv"

N_ITER     = 1000           # bootstrap iterations per association type
random.seed(None)           # set an int for reproducibility

## 2. Load DrugMechDB

In [2]:
with open(DMDB_YAML) as fh:
    ind = yaml.safe_load(fh)

# Node table: id / name / label  (strip stray unicode BOM)
clean = lambda s: s.replace("\ufeff", "")
dmdb_nodes = pd.DataFrame(
    [{"id": clean(n["id"]), "name": clean(n["name"]), "label": clean(n["label"])}
     for p in ind for n in p["nodes"]]
)
dmdb_all_nodes = set(dmdb_nodes["id"].unique())
print(f"DMDB: {len(ind)} indication paths, {len(dmdb_all_nodes)} unique nodes")

DMDB: 4781 indication paths, 5119 unique nodes


## 3. Load MechRepoNet (MIND)

In [3]:
mrn_nodes = pd.read_csv(MRN_NODES)
mrn_edges = pd.read_csv(MRN_EDGES, low_memory=False)

# All directed MRN edges as a fast lookup set
mrn_tups = set(zip(mrn_edges["start_id"], mrn_edges["end_id"]))
print(f"MRN: {len(mrn_nodes)} nodes, {len(mrn_edges)} edges")

/tmp/ipykernel_3750813/1393993683.py:1: DtypeWarning: Columns (4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  mrn_nodes = pd.read_csv(MRN_NODES)


MRN: 250035 nodes, 9652116 edges


## 4. Map DMDB node IDs to MRN node IDs

A DMDB edge counts as "in MRN" only if both of its nodes map to MRN nodes. We map in three passes:
1. **Direct** — the DMDB id is itself an MRN id.
2. **Xref** — an MRN node lists the DMDB id in its `xrefs`.
3. **Name string-match** — exact-ish name match within the equivalent MRN node type.

In [7]:
# Inlined from data_tools.wiki.get_curi_xrefs (avoids the heavy data_tools/scipy import)
def get_xrefs(nodes):
    out = nodes[["id", "xrefs"]].dropna(subset=["xrefs"]).copy()
    out["xrefs"] = out["xrefs"].str.split("|")
    return out.explode("xrefs")

def get_curi_xrefs(nodes, curi):
    out = get_xrefs(nodes)
    return out[out["xrefs"].str.startswith(curi + ":")].copy()

NODE_COLS = ["id", "name", "label"]

# --- Pass 1: direct id match ---
node_info = (dmdb_nodes.drop_duplicates(subset=["id"], keep="last")
                       .rename(columns={c: "dmdb_" + c for c in NODE_COLS}))
shared = mrn_nodes.query("id in @dmdb_all_nodes")[NODE_COLS]
node_info = node_info.merge(shared.rename(columns={c: "mrn_" + c for c in NODE_COLS}),
                            left_on="dmdb_id", right_on="mrn_id", how="left")

In [8]:
# --- Pass 2: xref match (only for MRN nodes whose id is NOT a DMDB id) ---
dmdb_curis = {nid.split(":")[0] for nid in dmdb_all_nodes}
unmatched_mrn = mrn_nodes.query("id not in @dmdb_all_nodes")

curi_xrefs = pd.concat([get_curi_xrefs(unmatched_mrn, c) for c in dmdb_curis])
shared_xrefd = curi_xrefs.query("xrefs in @dmdb_all_nodes")   # cols: id (mrn), xrefs (dmdb)

tmp = node_info[node_info["mrn_id"].isnull()].drop(columns=["mrn_" + c for c in NODE_COLS])
tmp = tmp.merge(shared_xrefd.rename(columns={"xrefs": "dmdb_id", "id": "mrn_id"}),
                on="dmdb_id", how="left")
tmp = tmp.merge(mrn_nodes[NODE_COLS].rename(columns={c: "mrn_" + c for c in NODE_COLS}),
                on="mrn_id", how="left")
node_info = pd.concat([node_info.dropna(subset=["mrn_id"]), tmp])

In [9]:
# --- Pass 3: string / name match within the equivalent MRN type ---
DMDB_TO_MRN_TYPE = {
    "Drug": "ChemicalSubstance", "Protein": "MacromolecularMachine", "Disease": "Disease",
    "BiologicalProcess": "BiologicalProcessOrActivity", "Pathway": "Pathway",
    "ChemicalSubstance": "ChemicalSubstance", "GrossAnatomicalStructure": "AnatomicalEntity",
    "MolecularActivity": "BiologicalProcessOrActivity", "OrganismTaxon": "OrganismTaxon",
    "GeneFamily": "GeneFamily", "CellularComponent": "AnatomicalEntity",
    "PhenotypicFeature": "PhenotypicFeature", "Cell": "AnatomicalEntity",
    "MacromolecularComplex": "MacromolecularMachine",
}

unmapped = node_info[node_info["mrn_id"].isnull()]
matches = []
for row in unmapped.itertuples():
    lower = row.dmdb_name.lower()
    mrn_type = DMDB_TO_MRN_TYPE.get(row.dmdb_label)   # resolve type in Python (avoid nested @ in query)
    cands = mrn_nodes[mrn_nodes["label"] == mrn_type]
    hit = cands[cands["name"].str.lower().apply(lambda s: lower in str(s))]
    if len(hit) == 1:                       # accept only unambiguous single hits
        hit = hit.iloc[[0]][NODE_COLS].copy()
        hit["dmdb_id"] = row.dmdb_id
        matches.append(hit)

if matches:
    matches = pd.concat(matches)
    found = set(matches["dmdb_id"])
    filled = (node_info.query("dmdb_id in @found")[["dmdb_" + c for c in NODE_COLS]]
              .merge(matches.rename(columns={**{c: "mrn_" + c for c in NODE_COLS}}),
                     on="dmdb_id", how="left"))
    node_info = pd.concat([node_info.query("dmdb_id not in @found"), filled]).reset_index(drop=True)

n_mapped = node_info.dropna(subset=["mrn_id"]).drop_duplicates("dmdb_id").shape[0]
n_total  = node_info.drop_duplicates("dmdb_id").shape[0]
print(f"{n_mapped}/{n_total} DMDB nodes mapped to MRN ({n_mapped/n_total:.1%})")

4159/5119 DMDB nodes mapped to MRN (81.2%)


## 5. Mark which DMDB edges exist in MRN

An edge is "in MRN" if its mapped (start, end) pair appears in MRN in **either direction**.

In [10]:
dmdb_edges = pd.DataFrame(
    [{"dmdb_start_id": clean(e["source"]), "dmdb_end_id": clean(e["target"])}
     for p in ind for e in p["links"]]
).drop_duplicates()

# Attach mapped MRN ids
id_map = node_info[["dmdb_id", "mrn_id"]]
edge_info = (dmdb_edges
    .merge(id_map.rename(columns={"dmdb_id": "dmdb_start_id", "mrn_id": "mrn_start_id"}),
           on="dmdb_start_id", how="left")
    .merge(id_map.rename(columns={"dmdb_id": "dmdb_end_id", "mrn_id": "mrn_end_id"}),
           on="dmdb_end_id", how="left"))

fwd = [(s, e) in mrn_tups for s, e in zip(edge_info["mrn_start_id"], edge_info["mrn_end_id"])]
rev = [(e, s) in mrn_tups for s, e in zip(edge_info["mrn_start_id"], edge_info["mrn_end_id"])]
edge_info["in_mrn"] = np.array(fwd) | np.array(rev)

# Set of observed DMDB pairs that are present in MRN
match_pairs = set(zip(edge_info.loc[edge_info["in_mrn"], "dmdb_start_id"],
                      edge_info.loc[edge_info["in_mrn"], "dmdb_end_id"]))
print(f"{edge_info['in_mrn'].sum()}/{len(edge_info)} unique DMDB edges found in MRN")

3164/10985 unique DMDB edges found in MRN


## 6. Group DMDB edges by concept-type association

In [11]:
# id -> label lookup (DMDB)
dmdb_label = dict(zip(dmdb_nodes["id"], dmdb_nodes["label"]))

association_types = defaultdict(list)
for p in ind:
    node_label = {n["id"]: n["label"] for n in p["nodes"]}
    for link in p["links"]:
        s, t = link["source"], link["target"]
        rel = f'{node_label[s]}_{node_label[t]}'
        association_types[rel].append((s, t))

# Most frequent association types first
assoc_order = sorted(association_types, key=lambda k: len(association_types[k]), reverse=True)
print(f"{len(assoc_order)} association types; top 5:")
for a in assoc_order[:5]:
    print(f"  {a:40s} {len(association_types[a])}")

129 association types; top 5:
  Protein_BiologicalProcess                4704
  Drug_Protein                             4455
  BiologicalProcess_BiologicalProcess      2835
  Protein_Protein                          2159
  BiologicalProcess_Disease                1874


## 7. Bootstrap

For each association type:
- **Observed**: % of its (head, tail) pairs that are in `match_pairs`.
- **Null**: re-pair random heads with random tails (drawn from that type's own heads/tails),
  then measure the % of the random edges present in MRN (forward direction, matching the original).
- Repeat `N_ITER` times → mean, 99% CI, and empirical p-value.

In [12]:
def random_assoc(pairs):
    """Build up to len(pairs) unique random (head, tail) edges from this type's heads & tails."""
    pairs = list(set(pairs))
    heads = [h for h, _ in pairs]
    tails = [t for _, t in pairs]
    target_n = len(pairs)
    out, attempts = set(), 0
    while len(out) < target_n and attempts < 10000:
        out.add((random.choice(heads), random.choice(tails)))
        attempts += 1
    return list(out)

# Pre-rename mapping frames once (used inside the loop)
start_map = id_map.rename(columns={"dmdb_id": "dmdb_start_id", "mrn_id": "mrn_start_id"})
end_map   = id_map.rename(columns={"dmdb_id": "dmdb_end_id", "mrn_id": "mrn_end_id"})

def bootstrap_assoc(pairs, n_iter=N_ITER):
    # observed overlap %
    uniq = set(pairs)
    expec_match = len(uniq & match_pairs) * 100 / len(uniq)

    dist = []
    for _ in range(n_iter):
        rnd = pd.DataFrame(random_assoc(pairs), columns=["dmdb_start_id", "dmdb_end_id"])
        rnd = (rnd.merge(start_map, on="dmdb_start_id", how="left")
                  .merge(end_map,   on="dmdb_end_id",   how="left")
                  .drop_duplicates(subset=["dmdb_start_id", "dmdb_end_id"]))
        in_mrn = [(s, e) in mrn_tups for s, e in zip(rnd["mrn_start_id"], rnd["mrn_end_id"])]
        dist.append(np.sum(in_mrn) * 100 / len(rnd))

    dist = np.array(dist)
    ci = np.percentile(dist, [1, 99])
    p_val = np.mean(np.abs(dist) > np.abs(expec_match))
    return expec_match, dist.mean(), ci, p_val

## 8. Run and save

In [13]:
rows = []
for a in tqdm(assoc_order, desc="association types"):
    expec, mean_bs, ci, p = bootstrap_assoc(association_types[a])
    rows.append({
        "association": a,
        "dmdb_count": len(association_types[a]),
        "expec_match": expec,
        "mean_bootstrap": mean_bs,
        "ci_low": ci[0],
        "ci_high": ci[1],
        "p_val": p,
    })

results = pd.DataFrame(rows)
results.to_csv(OUT_CSV, index=False)
print(f"Wrote {len(results)} rows -> {OUT_CSV}")
results.head(10)

association types: 100%|██████████| 129/129 [08:41<00:00,  4.04s/it]

Wrote 129 rows -> random_assoc_eval_dmdb_mrn_v1_regenerated.csv


,association,dmdb_count,expec_match,mean_bootstrap,ci_low,ci_high,p_val
0,Protein_BiologicalProcess,4704,41.257367,2.329470,1.375246,3.438114,0.000
1,Drug_Protein,4455,61.322151,4.803305,3.700049,5.870745,0.000
2,BiologicalProcess_BiologicalProcess,2835,0.404040,0.296364,0.000000,1.010101,0.179
3,Protein_Protein,2159,1.190476,0.011905,0.000000,0.011905,0.000
4,BiologicalProcess_Disease,1874,56.277056,40.207468,36.579004,43.939394,0.000
5,PhenotypicFeature_Disease,1366,6.737589,0.003369,0.000000,0.177305,0.000
6,OrganismTaxon_Disease,1338,31.283422,1.766845,0.534759,3.208556,0.000
7,BiologicalProcess_OrganismTaxon,1149,12.328767,7.323014,4.657534,10.410959,0.000
8,BiologicalProcess_PhenotypicFeature,1139,25.595238,20.419345,16.071429,25.000000,0.005
9,ChemicalSubstance_BiologicalProcess,976,10.000000,3.997727,1.363636,7.272727,0.001
